# Challenge Dataset Evaluation
Evaluates A1 (Logistic Regression) and A2 (DistilBERT) on the SRL challenge dataset.

**Metrics:**
- MFT caps (1, 3, 4): per-instance failure rate
- INV caps (2, 5, 6): per-instance failure rate + pair inconsistency rate + both-wrong rate

## 1. Setup

In [8]:
import os, json
import pandas as pd
import joblib
import torch
from transformers import AutoModelForTokenClassification, DistilBertTokenizerFast
import sys

HERE         = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(HERE, ".."))

from a1.train import predict_srl as predict_srl_a1
from a2.srl_model import predict_srl as predict_srl_a2

os.makedirs('results', exist_ok=True)
print('Setup complete.')

Setup complete.


## 2. Load Models

In [12]:
# ── A1: Logistic Regression ──
a1_model      = joblib.load(os.path.join(PROJECT_ROOT, "a1", "model", "model.joblib"))
a1_vectorizer = joblib.load(os.path.join(PROJECT_ROOT, "a1", "model", "vectorizer.joblib"))

print('A1 loaded.')

# ── A2: DistilBERT ──
A2_CHECKPOINT = os.path.join(PROJECT_ROOT, "a2", "srl-distilbert-model")  # folder containing config.json, pytorch_model.bin, tokenizer files


a2_model     = AutoModelForTokenClassification.from_pretrained(A2_CHECKPOINT)
a2_tokenizer = DistilBertTokenizerFast.from_pretrained(A2_CHECKPOINT)
id2label     = a2_model.config.id2label

device = torch.device('mps' if torch.backends.mps.is_available()
                       else 'cuda' if torch.cuda.is_available()
                       else 'cpu')
a2_model = a2_model.to(device)
a2_model.eval()

print(f'A2 loaded. Device: {device}')
print(f'Label vocabulary ({len(id2label)} labels): {list(id2label.values())[:10]} ...')

A1 loaded.


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 8872.78it/s]


A2 loaded. Device: mps
Label vocabulary (60 labels): ['O', 'ARG0', 'ARG1', 'ARG1-DSP', 'ARG2', 'ARG3', 'ARG4', 'ARG5', 'ARGA', 'ARGM-ADJ'] ...


## 3. Load Challenge Dataset

In [13]:
DATASET_DIR = 'challenge_dataset'

files = {
    'cap1': 'cap1_core_arguments.json',
    'cap2': 'cap2_voice_alternation.json',
    'cap3': 'cap3_pp_attachment.json',
    'cap4': 'cap4_argm_gol.json',
    'cap5': 'cap5_spray_load.json',
    'cap6': 'cap6_entity_substitution.json',
}

dataset = {}
for cap, fname in files.items():
    with open(os.path.join(DATASET_DIR, fname)) as f:
        dataset[cap] = json.load(f)
    print(f'{cap}: {len(dataset[cap])} instances')

cap1: 20 instances
cap2: 20 instances
cap3: 15 instances
cap4: 10 instances
cap5: 20 instances
cap6: 20 instances


## 4. Inference Helpers

In [14]:
def get_pred_a1(instance):
    result = predict_srl_a1(
        instance['sentence'],
        instance['predicate'],
        a1_model,
        a1_vectorizer
    )
    # result is a list of (token, label) tuples
    return result[instance['target_index']][1]


def get_pred_a2(instance):
    result = predict_srl_a2(
        instance['sentence'],
        instance['predicate'],
        a2_model,
        a2_tokenizer,
        id2label,
        device
    )
    return result[instance['target_index']][1]


def run_capability(instances, cap_name):
    rows = []
    for inst in instances:
        pred_a1 = get_pred_a1(inst)
        pred_a2 = get_pred_a2(inst)
        rows.append({
            'capability':   cap_name,
            'id':           inst['id'],
            'type':         inst['type'],
            'test':         inst['test'],
            'pair_id':      inst.get('pair_id', ''),
            'role':         inst.get('role', ''),
            'sentence':     ' '.join(inst['sentence']),
            'target_token': inst['sentence'][inst['target_index']],
            'target_label': inst['target_label'],
            'pred_a1':      pred_a1,
            'pred_a2':      pred_a2,
            'pass_a1':      pred_a1 == inst['target_label'],
            'pass_a2':      pred_a2 == inst['target_label'],
        })
    return pd.DataFrame(rows)


print('Helpers ready.')

Helpers ready.


## 5. Run Evaluation

In [15]:
results = {}
all_rows = []

for cap, instances in dataset.items():
    print(f'Running {cap} ({len(instances)} instances)...')
    df = run_capability(instances, cap)
    results[cap] = df
    all_rows.append(df)

all_results = pd.concat(all_rows, ignore_index=True)
print(f'Done. {len(all_results)} total predictions per model.')

Running cap1 (20 instances)...
Running cap2 (20 instances)...
Running cap3 (15 instances)...
Running cap4 (10 instances)...
Running cap5 (20 instances)...
Running cap6 (20 instances)...
Done. 105 total predictions per model.


## 6. Results

### 6a. Per-instance predictions (spot check)

In [18]:
# Print first few rows per capability so you can sanity-check individual predictions
for cap, df in results.items():
    print(f'\n{cap}:')
    cols = ['id', 'role', 'target_token', 'target_label', 'pred_a1', 'pass_a1', 'pred_a2', 'pass_a2']
    print(df[cols].to_string(index=False))


cap1:
       id role target_token target_label pred_a1  pass_a1 pred_a2  pass_a2
cap1a_001         committee         ARG0    ARG0     True    ARG0     True
cap1b_001            policy         ARG1    ARG1     True    ARG1     True
cap1a_002             swarm         ARG0    ARG0     True    ARG0     True
cap1b_002          intruder         ARG1    ARG1     True    ARG1     True
cap1a_003          engineer         ARG0    ARG1    False    ARG0     True
cap1b_003         prototype         ARG1       O    False       O    False
cap1a_004           teacher         ARG0    ARG0     True    ARG0     True
cap1b_004           concept         ARG1    ARG1     True    ARG1     True
cap1a_005               dog         ARG0    ARG0     True    ARG0     True
cap1b_005              ball         ARG1    ARG1     True    ARG1     True
cap1a_006         scientist         ARG0    ARG0     True    ARG0     True
cap1b_006              data         ARG1    ARG1     True    ARG1     True
cap1a_007       fi

### 6b. Per-instance failure rates

In [19]:
def failure_rate(series):
    return round(1 - series.mean(), 3)


summary_rows = []
for cap, df in results.items():
    for test_name, group in df.groupby('test'):
        summary_rows.append({
            'capability':       cap,
            'test':             test_name,
            'type':             group['type'].iloc[0],
            'n':                len(group),
            'failure_rate_a1':  failure_rate(group['pass_a1']),
            'failure_rate_a2':  failure_rate(group['pass_a2']),
        })

summary = pd.DataFrame(summary_rows)
print('Per-instance failure rates (0.0 = all pass, 1.0 = all fail)')
print('=' * 75)
print(summary.to_string(index=False))

Per-instance failure rates (0.0 = all pass, 1.0 = all fail)
capability                   test type  n  failure_rate_a1  failure_rate_a2
      cap1              core_arg0  MFT 10             0.10             0.00
      cap1              core_arg1  MFT 10             0.10             0.10
      cap2      voice_alternation  INV 20             0.05             0.00
      cap3     pp_attachment_arg2  MFT  5             0.00             0.20
      cap3      pp_attachment_com  MFT  5             0.80             0.20
      cap3      pp_attachment_mnr  MFT  5             1.00             0.00
      cap4     argm_gol_detection  MFT 10             1.00             1.00
      cap5 spray_load_alternation  INV 20             0.15             0.05
      cap6    entity_substitution  INV 20             0.00             0.00


### 6c. Pair-level analysis (INV capabilities only)

In [20]:
inv_caps = [cap for cap, df in results.items() if df['type'].iloc[0] == 'INV']
pair_rows = []

for cap in inv_caps:
    df = results[cap]
    for pair_id, group in df.groupby('pair_id'):
        if len(group) != 2:
            continue
        a = group.iloc[0]
        b = group.iloc[1]
        pair_rows.append({
            'capability':    cap,
            'test':          a['test'],
            'pair_id':       pair_id,
            'role_a':        a['role'],
            'role_b':        b['role'],
            'sentence_a':    a['sentence'],
            'sentence_b':    b['sentence'],
            'target_label':  a['target_label'],
            'pred_a1_a':     a['pred_a1'],
            'pred_a1_b':     b['pred_a1'],
            'pass_a1_a':     a['pass_a1'],
            'pass_a1_b':     b['pass_a1'],
            'consistent_a1': a['pred_a1'] == b['pred_a1'],
            'both_wrong_a1': (not a['pass_a1']) and (not b['pass_a1']),
            'pred_a2_a':     a['pred_a2'],
            'pred_a2_b':     b['pred_a2'],
            'pass_a2_a':     a['pass_a2'],
            'pass_a2_b':     b['pass_a2'],
            'consistent_a2': a['pred_a2'] == b['pred_a2'],
            'both_wrong_a2': (not a['pass_a2']) and (not b['pass_a2']),
        })

pairs_df = pd.DataFrame(pair_rows)

# Print detailed pair breakdown
for cap in inv_caps:
    cap_pairs = pairs_df[pairs_df['capability'] == cap]
    print(f'\n{cap} — {cap_pairs["test"].iloc[0]}')
    print('-' * 90)
    cols = ['pair_id', 'target_label',
            'pred_a1_a', 'pass_a1_a', 'pred_a1_b', 'pass_a1_b', 'consistent_a1',
            'pred_a2_a', 'pass_a2_a', 'pred_a2_b', 'pass_a2_b', 'consistent_a2']
    print(cap_pairs[cols].to_string(index=False))
    print()
    n = len(cap_pairs)
    print(f'  Inconsistency rate  A1: {round(1 - cap_pairs["consistent_a1"].mean(), 3)}  '
          f'A2: {round(1 - cap_pairs["consistent_a2"].mean(), 3)}')
    print(f'  Both-wrong rate     A1: {round(cap_pairs["both_wrong_a1"].mean(), 3)}  '
          f'A2: {round(cap_pairs["both_wrong_a2"].mean(), 3)}')


cap2 — voice_alternation
------------------------------------------------------------------------------------------
 pair_id target_label pred_a1_a  pass_a1_a pred_a1_b  pass_a1_b  consistent_a1 pred_a2_a  pass_a2_a pred_a2_b  pass_a2_b  consistent_a2
cap2_001         ARG0      ARG0       True      ARG0       True           True      ARG0       True      ARG0       True           True
cap2_002         ARG0      ARG0       True      ARG0       True           True      ARG0       True      ARG0       True           True
cap2_003         ARG0      ARG0       True      ARG0       True           True      ARG0       True      ARG0       True           True
cap2_004         ARG0      ARG0       True      ARG0       True           True      ARG0       True      ARG0       True           True
cap2_005         ARG0      ARG0       True      ARG0       True           True      ARG0       True      ARG0       True           True
cap2_006         ARG0      ARG0       True      ARG0       True    

### 6d. Master overview table

In [21]:
# Clean overview suitable for the report
print(f'{"Test":<35} {"Type":<5} {"N":>4}  {"Fail A1":>8}  {"Fail A2":>8}')
print('-' * 65)
for _, row in summary.iterrows():
    test = row['test'].replace('_', ' ')
    print(f'  {test:<33} {row["type"]:<5} {row["n"]:>4}  '
          f'{row["failure_rate_a1"]:>8.3f}  {row["failure_rate_a2"]:>8.3f}')

if len(pairs_df) > 0:
    print()
    print(f'{"Pair inconsistency (INV only)":<35} {"Type":<5} {"N":>4}  {"Incon A1":>8}  {"Incon A2":>8}')
    print('-' * 65)
    for cap in inv_caps:
        cap_pairs = pairs_df[pairs_df['capability'] == cap]
        test = cap_pairs['test'].iloc[0].replace('_', ' ')
        inc_a1 = round(1 - cap_pairs['consistent_a1'].mean(), 3)
        inc_a2 = round(1 - cap_pairs['consistent_a2'].mean(), 3)
        print(f'  {test:<33} {"INV":<5} {len(cap_pairs):>4}  '
              f'{inc_a1:>8.3f}  {inc_a2:>8.3f}')

Test                                Type     N   Fail A1   Fail A2
-----------------------------------------------------------------
  core arg0                         MFT     10     0.100     0.000
  core arg1                         MFT     10     0.100     0.100
  voice alternation                 INV     20     0.050     0.000
  pp attachment arg2                MFT      5     0.000     0.200
  pp attachment com                 MFT      5     0.800     0.200
  pp attachment mnr                 MFT      5     1.000     0.000
  argm gol detection                MFT     10     1.000     1.000
  spray load alternation            INV     20     0.150     0.050
  entity substitution               INV     20     0.000     0.000

Pair inconsistency (INV only)       Type     N  Incon A1  Incon A2
-----------------------------------------------------------------
  voice alternation                 INV     10     0.100     0.000
  spray load alternation            INV     10     0.800     1.

## 7. Save Outputs

In [22]:
# All predictions combined
cols = ['capability', 'id', 'type', 'test', 'pair_id', 'role',
        'sentence', 'target_token', 'target_label',
        'pred_a1', 'pass_a1', 'pred_a2', 'pass_a2']
all_results[cols].to_csv('results/all_predictions.tsv', sep='\t', index=False)

# Per-model files
a1_cols = ['capability', 'id', 'type', 'test', 'pair_id', 'role',
           'sentence', 'target_token', 'target_label', 'pred_a1', 'pass_a1']
a2_cols = ['capability', 'id', 'type', 'test', 'pair_id', 'role',
           'sentence', 'target_token', 'target_label', 'pred_a2', 'pass_a2']
all_results[a1_cols].to_csv('results/a1_predictions.tsv', sep='\t', index=False)
all_results[a2_cols].to_csv('results/a2_predictions.tsv', sep='\t', index=False)

# Summary tables
summary.to_csv('results/summary_failure_rates.tsv', sep='\t', index=False)
if len(pairs_df) > 0:
    pairs_df.to_csv('results/pair_analysis.tsv', sep='\t', index=False)

print('Saved to results/:')
for f in sorted(os.listdir('results')):
    print(f'  {f}')

Saved to results/:
  a1_predictions.tsv
  a2_predictions.tsv
  all_predictions.tsv
  label_plot.png
  pair_analysis.tsv
  summary_failure_rates.tsv
